In [ ]:

import os
import pandas as pd
import matplotlib.pyplot as plt

data_path = "../data/csv"
stats_preview = pd.read_csv(os.path.join(data_path, "other_stats.csv"), nrows=5)
games_preview = pd.read_csv(os.path.join(data_path, "game.csv"), nrows=5)

In [ ]:
stats_preview = pd.read_csv("../data/csv/other_stats.csv", nrows=5)
stats_preview

In [ ]:
games_preview.columns

In [ ]:
games_preview = pd.read_csv("../data/csv/game.csv", nrows=5)
games_preview
# step 2

In [ ]:
import pandas as pd
import os

data_path = "../data/csv"

for file in os.listdir(data_path):
    file_path = os.path.join(data_path, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(file, round(size_mb, 2), "MB")

    #step 1: load the data, this part is from here and go up, I dont know why when I try create new code blocks it is appearing from the top, and push it down to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

games = pd.read_csv("../data/csv/game.csv")

# Standardize column names
games.columns = games.columns.str.lower().str.strip().str.replace(" ", "_")

# Keep only regular season games
rows_before_filter = len(games)
regular_season_mask = (
    games["season_type"].astype("string").str.strip().str.casefold()
    == "regular season"
)
games = games.loc[regular_season_mask].copy()

print("Rows before season filter:", rows_before_filter)
print("Regular season rows:", len(games))
print("Non regular season rows removed:", rows_before_filter - len(games))

games.head()

In [ ]:
before = games.shape[0]
games = games.drop_duplicates()
after = games.shape[0]

print("Rows before:", before)
print("Rows after:", after)
print("Duplicates removed:", before - after)

# Cleaning 1: remove duplicates

In [ ]:
games["game_date"] = pd.to_datetime(games["game_date"])

games[["game_id", "game_date"]].head()

# Cleaning 2: convert date

In [ ]:
home = games[[
    "game_id", "game_date", "season_id",
    "team_id_home", "team_name_home", "wl_home",
    "pts_home", "fg_pct_home", "reb_home", "ast_home", "tov_home"
]].copy()

home.columns = [
    "game_id", "game_date", "season_id",
    "team_id", "team_name", "win_loss",
    "pts", "fg_pct", "reb", "ast", "tov"
]

home["home_away"] = "Home"


away = games[[
    "game_id", "game_date", "season_id",
    "team_id_away", "team_name_away", "wl_away",
    "pts_away", "fg_pct_away", "reb_away", "ast_away", "tov_away"
]].copy()

away.columns = [
    "game_id", "game_date", "season_id",
    "team_id", "team_name", "win_loss",
    "pts", "fg_pct", "reb", "ast", "tov"
]

away["home_away"] = "Away"


team_games = pd.concat([home, away], ignore_index=True)

team_games.head()

# Cleaning 3: create home-team rows and away-team rows

In [ ]:
team_games = team_games.sort_values(["team_id", "game_date"])

team_games.head()

# Cleaning 4: sort games chronologically by team

In [ ]:
team_games["previous_game_date"] = team_games.groupby("team_id")["game_date"].shift(1)

team_games["rest_days"] = (
    team_games["game_date"] - team_games["previous_game_date"]
).dt.days - 1

team_games[["team_name", "game_date", "previous_game_date", "rest_days"]].head(10)
# Cleaning 5: create rest_days

In [ ]:
def categorize_rest(days):
    if pd.isna(days):
        return "First Game"
    elif days <= 0:
        return "0 Days"
    elif days == 1:
        return "1 Day"
    elif days == 2:
        return "2 Days"
    else:
        return "3+ Days"

team_games["rest_category"] = team_games["rest_days"].apply(categorize_rest)

team_games["rest_category"].value_counts()

# Cleaning 6: create rest categories

In [ ]:
team_games["win"] = team_games["win_loss"].map({"W": 1, "L": 0})

team_games[["team_name", "win_loss", "win"]].head()

# Create win column

In [ ]:
summary_stats = team_games[["pts", "fg_pct", "reb", "ast", "tov", "rest_days"]].describe()

summary_stats

# EDA 1: Summary statistics

In [ ]:
eda_df = team_games[team_games["rest_category"] != "First Game"].copy()

plt.figure(figsize=(8, 5))
plt.hist(eda_df["rest_days"].dropna(), bins=20)
plt.title("Distribution of Rest Days")
plt.xlabel("Rest Days")
plt.ylabel("Number of Team-Games")
plt.show()

# EDA 2: Distribution of rest days

In [ ]:
points_by_rest = eda_df.groupby("rest_category")["pts"].mean().reindex(
    ["0 Days", "1 Day", "2 Days", "3+ Days"]
)

points_by_rest
# EDA 3: Average points by rest category

In [ ]:
plt.figure(figsize=(8, 5))
points_by_rest.plot(kind="bar")
plt.title("Average Points by Rest Category")
plt.xlabel("Rest Category")
plt.ylabel("Average Points")
plt.xticks(rotation=0)
plt.show()
# make the bar chart EDA 3

In [ ]:
fg_by_rest = eda_df.groupby("rest_category")["fg_pct"].mean().reindex(
    ["0 Days", "1 Day", "2 Days", "3+ Days"]
)

fg_by_rest
# EDA 4: Shooting percentage by rest category

In [ ]:
plt.figure(figsize=(8, 5))
fg_by_rest.plot(kind="bar")
plt.title("Average Field Goal Percentage by Rest Category")
plt.xlabel("Rest Category")
plt.ylabel("Average FG%")
plt.xticks(rotation=0)
plt.show()
# make the bar chart EDA 4

In [ ]:
win_by_rest = eda_df.groupby("rest_category")["win"].mean().reindex(
    ["0 Days", "1 Day", "2 Days", "3+ Days"]
) * 100

win_by_rest

# EDA 5: Win percentage by rest category

In [ ]:
plt.figure(figsize=(8, 5))
win_by_rest.plot(kind="bar")
plt.title("Win Percentage by Rest Category")
plt.xlabel("Rest Category")
plt.ylabel("Win Percentage (%)")
plt.xticks(rotation=0)
plt.show()
# make the bar chart EDA 5

In [ ]:
corr = eda_df[["rest_days", "pts", "fg_pct", "reb", "ast", "tov", "win"]].corr()

corr

# EDA 6: Correlation analysis

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Between Rest Days and Performance Statistics")

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, round(corr.iloc[i, j], 2), ha="center", va="center")

plt.tight_layout()
plt.show()
# turns that table into a heatmap figure

In [ ]:
team_games.to_csv("../data/cleaned_team_rest_data.csv", index=False)

# save cleaned data